## Setup
Conexão com DuckDB e carregamento das tabelas Olist a partir dos CSVs originais.

In [143]:
import duckdb

con = duckdb.connect()

data_path = "../data/raw"

con.execute(f"""
    CREATE TABLE orders AS SELECT * FROM read_csv_auto('{data_path}/olist_orders_dataset.csv');
    CREATE TABLE order_items AS SELECT * FROM read_csv_auto('{data_path}/olist_order_items_dataset.csv');
    CREATE TABLE customers AS SELECT * FROM read_csv_auto('{data_path}/olist_customers_dataset.csv');
    CREATE TABLE products AS SELECT * FROM read_csv_auto('{data_path}/olist_products_dataset.csv');
    CREATE TABLE sellers AS SELECT * FROM read_csv_auto('{data_path}/olist_sellers_dataset.csv');
""")

print(con.execute("SHOW TABLES").fetchall())


[('customers',), ('order_items',), ('orders',), ('products',), ('sellers',)]


In [144]:
con.execute("DESCRIBE orders").df()


,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


In [145]:
con.execute("DESCRIBE order_items").df()


,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,order_item_id,BIGINT,YES,None,None,None
2,product_id,VARCHAR,YES,None,None,None
3,seller_id,VARCHAR,YES,None,None,None
4,shipping_limit_date,TIMESTAMP,YES,None,None,None
5,price,DOUBLE,YES,None,None,None
6,freight_value,DOUBLE,YES,None,None,None


In [146]:
con.execute("DESCRIBE customers").df()


,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,YES,None,None,None
1,customer_unique_id,VARCHAR,YES,None,None,None
2,customer_zip_code_prefix,VARCHAR,YES,None,None,None
3,customer_city,VARCHAR,YES,None,None,None
4,customer_state,VARCHAR,YES,None,None,None


In [147]:
con.execute("DESCRIBE products").df()


,column_name,column_type,null,key,default,extra
0,product_id,VARCHAR,YES,None,None,None
1,product_category_name,VARCHAR,YES,None,None,None
2,product_name_lenght,BIGINT,YES,None,None,None
3,product_description_lenght,BIGINT,YES,None,None,None
4,product_photos_qty,BIGINT,YES,None,None,None
5,product_weight_g,BIGINT,YES,None,None,None
6,product_length_cm,BIGINT,YES,None,None,None
7,product_height_cm,BIGINT,YES,None,None,None
8,product_width_cm,BIGINT,YES,None,None,None


In [148]:
con.execute("DESCRIBE sellers").df()


,column_name,column_type,null,key,default,extra
0,seller_id,VARCHAR,YES,None,None,None
1,seller_zip_code_prefix,VARCHAR,YES,None,None,None
2,seller_city,VARCHAR,YES,None,None,None
3,seller_state,VARCHAR,YES,None,None,None


## 1. Volume de pedidos por estado
Total de pedidos agrupado por estado do cliente, ordenado do maior pro menor.

In [149]:
con.execute("""
    SELECT
        customer_state
        , COUNT(order_id) AS total_pedidos
    FROM 
        orders 
    JOIN 
        customers ON orders.customer_id = customers.customer_id 
    GROUP BY 
        customer_state 
    ORDER BY 
        total_pedidos DESC
""").df()


,customer_state,total_pedidos
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


## 2. Distribuição por status
Quantidade de pedidos por status, ordenado do maior pro menor.

In [150]:
con.execute("""
    SELECT
        order_status
        , COUNT(order_id) AS total_pedidos
    FROM
        orders
    GROUP BY
        order_status
    ORDER BY
        total_pedidos DESC
""").df()


,order_status,total_pedidos
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


## 3. Ticket médio por estado
Valor médio do preço dos itens agrupado por estado do cliente.

In [151]:
con.execute("""
    SELECT
        customer_state AS estado
        , AVG(price) AS ticket_medio
    FROM
        order_items
    JOIN
        orders ON order_items.order_id = orders.order_id
    JOIN 
        customers ON orders.customer_id = customers.customer_id
    GROUP BY 
        estado
    ORDER BY ticket_medio DESC
""").df()

,estado,ticket_medio
0,PB,191.475216
1,AL,180.889212
2,AC,173.727717
3,RO,165.973525
4,PA,165.692417
5,AP,164.320732
6,PI,160.358081
7,TO,157.529333
8,RN,156.965936
9,CE,153.758261


## 4. Top 10 produtos mais vendidos
Produtos com maior número de unidades vendidas na tabela order_items.

In [152]:
con.execute("""
    SELECT
        product_id
        , COUNT(*) AS quantidade_vendida
    FROM
        order_items
    GROUP BY
        product_id
    ORDER BY quantidade_vendida DESC
    LIMIT 10
""").df()


,product_id,quantidade_vendida
0,aca2eb7d00ea1a7b8ebd4e68314663af,527
1,99a4788cb24856965c36a24e339b6058,488
2,422879e10f46682990de24d770e7f83d,484
3,389d119b48cf3043d311335e499d9c6b,392
4,368c6c730842d78016ad823897a372db,388
5,53759a2ecddad2bb87a079a1f1519f73,373
6,d1c427060a0f73f6b889a5c7c61f2ac4,343
7,53b36df67ebb7c41585e8d54d6772e08,323
8,154e7e31ebfa092203795c972e5804a6,281
9,3dd2a17168ec895c781a9191c1e95ad7,274


## 5. Top 10 sellers por itens vendidos
Sellers com maior volume de itens vendidos, identificados pelo seller_id.

In [153]:
con.execute("""
    SELECT
        seller_id
        , COUNT(*) AS itens_vendidos
    FROM
        order_items
    GROUP BY 
        seller_id
    ORDER BY itens_vendidos DESC
    LIMIT 10
""").df()


,seller_id,itens_vendidos
0,6560211a19b47992c3666cc44a7e94c0,2033
1,4a3ca9315b744ce9f8e9374361493884,1987
2,1f50f920176fa81dab994f9023523100,1931
3,cc419e0650a3c5ba77189a1882b7556a,1775
4,da8622b14eb17ae2831f4ac5b9dab84a,1551
5,955fee9216a65b617aa5c0531780ce60,1499
6,1025f0e2d44d7041d6cf58b6550e0bfa,1428
7,7c67e1448b00f6e969d365cea6b010ab,1364
8,ea8482cd71df3c1969d7b9473ff13abc,1203
9,7a67c85e85bb2ce8582c35f2203ad736,1171
